# Fruit Classifier — PTH → TFLite + App-Dev Handover JSONs

This notebook performs all the following steps:
1. Converts `fruit_classifier_final.pth` to `fruit_classifier.tflite`
2. Generates `class_mapping.json` — fruit/condition label mapping
3. Generates `runtime_config.json` — input/output tensor specification, preprocessing
4. Generates `ood_config.json` *(optional section)* — OOD rejection threshold (Mahalanobis) — **this requires the original training dataset** (Kaggle dataset `ahmadfauzi89/fruitvision-mendeley`), so it is placed in a separate, clearly marked optional section.

The model expects a `(1, 3, 384, 384)` RGB tensor, ImageNet-normalized
(`mean=[0.485,0.456,0.406]`, `std=[0.229,0.224,0.225]`), and returns
`(fruit_logits[5], condition_logits[3], attention_map[14x14], features[1280])`.

**Instructions for running:** Always run Part 1 (Setup → TFLite convert → class_mapping/runtime_config) — no dataset is needed.
Only run Part 2 (OOD config) if the app development team requires `ood_config.json` — this requires Kaggle dataset access, so it takes time.

## Part 1 — Setup

In [1]:
# ── 1. Install dependencies ─────────────────────────────────────────
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu
!pip install -q ai-edge-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 18.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typeguard 4.5.2 requires typing_extensions>=4.14.0, but you have typing-extensions 4.12.2 which is incompatib

In [2]:
# ── 2. Upload fruit_classifier_final.pth ────────────────────────────
from google.colab import files
uploaded = files.upload()  # select fruit_classifier_final.pth
PTH_PATH = list(uploaded.keys())[0]
print('Using file:', PTH_PATH)

Saving fruit_classifier_final.pth to fruit_classifier_final.pth
Using file: fruit_classifier_final.pth


In [3]:
# ── 3. Model definition (exact architecture from neural-bites-v3-1) ─
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

IMG_SIZE = 384
FRUIT_NAMES     = ["Apple", "Banana", "Grape", "Mango", "Orange"]
CONDITION_NAMES = ["Formalin-mixed", "Fresh", "Rotten"]
NUM_FRUITS      = len(FRUIT_NAMES)
NUM_CONDITIONS  = len(CONDITION_NAMES)

class MultiHeadMobileNetV2(nn.Module):
    def __init__(self, num_fruits=5, num_conditions=3, freeze_backbone=True):
        super().__init__()
        base = models.mobilenet_v2(weights=None)  # weights loaded from .pth below
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)

        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad = False

        self.fruit_head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1280, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_fruits),
        )

        self.condition_head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1280, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_conditions),
        )

        self.attention_head = nn.Sequential(
            nn.Conv2d(1280, 64, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        feat_map = self.features(x)
        pooled = self.pool(feat_map)
        features = pooled.flatten(1)

        fruit_logits = self.fruit_head(features)
        condition_logits = self.condition_head(features)

        att_raw = self.attention_head(feat_map)
        att_map = F.interpolate(att_raw, size=(14, 14), mode='bilinear', align_corners=False)
        att_map = att_map.squeeze(1)

        return fruit_logits, condition_logits, att_map, features

In [4]:
# ── 4. Load the trained weights ─────────────────────────────────────
model = MultiHeadMobileNetV2(NUM_FRUITS, NUM_CONDITIONS, freeze_backbone=True)
state = torch.load(PTH_PATH, map_location='cpu')

if isinstance(state, dict) and 'model' in state and not any(
    k.startswith(('features.', 'fruit_head.', 'condition_head.', 'attention_head.'))
    for k in state.keys()
):
    state = state['model']

missing, unexpected = model.load_state_dict(state, strict=True)
print('Missing keys:', missing)
print('Unexpected keys:', unexpected)
model.eval()
print('✓ Weights loaded successfully')

Missing keys: []
Unexpected keys: []
✓ Weights loaded successfully


In [5]:
# ── 5. Sanity check: run a dummy forward pass ───────────────────────
with torch.no_grad():
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    fruit_logits, cond_logits, att_map, feats = model(dummy)
    print('fruit_logits:', fruit_logits.shape)
    print('condition_logits:', cond_logits.shape)
    print('attention_map:', att_map.shape)
    print('features:', feats.shape)

fruit_logits: torch.Size([1, 5])
condition_logits: torch.Size([1, 3])
attention_map: torch.Size([1, 14, 14])
features: torch.Size([1, 1280])


## Part 1b — Convert to TFLite

In [6]:
TFLITE_OUT = 'fruit_classifier.tflite'

try:
    import litert_torch
    sample_input = (torch.randn(1, 3, IMG_SIZE, IMG_SIZE),)
    edge_model = litert_torch.convert(model, sample_input)
    edge_model.export(TFLITE_OUT)
    print(f'✓ Saved {TFLITE_OUT} via litert-torch')
except ImportError:
    print('litert-torch not available or conversion failed — using ONNX -> onnx2tf fallback')
    !pip install -q onnx onnx2tf tensorflow
    import subprocess, os

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    onnx_path = 'fruit_classifier.onnx'
    torch.onnx.export(
        model, dummy, onnx_path,
        input_names=['input'],
        output_names=['fruit_logits', 'condition_logits', 'attention_map', 'features'],
        opset_version=13,
    )
    subprocess.run(['onnx2tf', '-i', onnx_path, '-o', 'saved_model_out', '-osd'], check=True)
    produced = os.path.join('saved_model_out', 'fruit_classifier_float32.tflite')
    if os.path.exists(produced):
        os.replace(produced, TFLITE_OUT)
        print(f'✓ Saved {TFLITE_OUT} via onnx2tf')
    else:
        print('Check saved_model_out/ for the produced .tflite file(s).')

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:03) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:06) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:02)

...

(00:25) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:25) [ DONE] LiteRT-Torch Convert (+00:25)

(00:00) [START] Write Model to fruit_classifier.tflite

(00:00) [ DONE] Write Model to fruit_classifier.tflite (+00:00)

✓ Saved fruit_classifier.tflite via litert-torch


In [7]:
# ── 7. Verify the TFLite model + read back actual tensor names/shapes ─
# (these are used below to build runtime_config.json so it always matches
# the real exported file, not assumptions)
import numpy as np
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=TFLITE_OUT)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print('Input details:', input_details)
print('Output details:', output_details)

x = np.random.randn(*input_details[0]['shape']).astype(input_details[0]['dtype'])
interpreter.set_tensor(input_details[0]['index'], x)
interpreter.invoke()
for od in output_details:
    print(od['name'], '->', interpreter.get_tensor(od['index']).shape)
print('✓ TFLite model runs correctly')

Input details: [{'name': 'serving_default_args_0', 'index': 0, 'shape': array([  1,   3, 384, 384], dtype=int32), 'shape_signature': array([  1,   3, 384, 384], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'serving_default_output_0_output', 'index': 194, 'shape': array([1, 5], dtype=int32), 'shape_signature': array([1, 5], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}, {'name': 'serving_default_output_1_output', 'index': 196, 'shape': array([1, 3], dtype=int32), 'shape_signature': array([1, 3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Part 1c — Generate `class_mapping.json` and `runtime_config.json`

These do not require a dataset — only the label list and tensor specifications (verified in Part 1).

In [8]:
# ── 8. class_mapping.json ────────────────────────────────────────────
import json, os

EXPORT_DIR = "mobile_deploy"
os.makedirs(EXPORT_DIR, exist_ok=True)

combined_classes_15way = sorted(
    f"{fruit}-{cond}" for fruit in FRUIT_NAMES for cond in CONDITION_NAMES
)

class_mapping = {
    "fruit_classes": {str(i): f for i, f in enumerate(FRUIT_NAMES)},
    "condition_classes": {str(i): c for i, c in enumerate(CONDITION_NAMES)},
    "combined_classes_15way": combined_classes_15way,
    "notes": {
        "combined_class_naming": "combined class = f'{fruit}-{condition}', split on the FIRST hyphen only (Formalin-mixed itself contains a hyphen)",
        "safety_critical_class": "condition index 0 (Formalin-mixed) is the safety-critical class"
    }
}
with open(os.path.join(EXPORT_DIR, "class_mapping.json"), "w") as f:
    json.dump(class_mapping, f, indent=2)
print("✓ class_mapping.json saved")
print(json.dumps(class_mapping, indent=2))

✓ class_mapping.json saved
{
  "fruit_classes": {
    "0": "Apple",
    "1": "Banana",
    "2": "Grape",
    "3": "Mango",
    "4": "Orange"
  },
  "condition_classes": {
    "0": "Formalin-mixed",
    "1": "Fresh",
    "2": "Rotten"
  },
  "combined_classes_15way": [
    "Apple-Formalin-mixed",
    "Apple-Fresh",
    "Apple-Rotten",
    "Banana-Formalin-mixed",
    "Banana-Fresh",
    "Banana-Rotten",
    "Grape-Formalin-mixed",
    "Grape-Fresh",
    "Grape-Rotten",
    "Mango-Formalin-mixed",
    "Mango-Fresh",
    "Mango-Rotten",
    "Orange-Formalin-mixed",
    "Orange-Fresh",
    "Orange-Rotten"
  ],
  "notes": {
    "combined_class_naming": "combined class = f'{fruit}-{condition}', split on the FIRST hyphen only (Formalin-mixed itself contains a hyphen)",
    "safety_critical_class": "condition index 0 (Formalin-mixed) is the safety-critical class"
  }
}


In [9]:
# ── 9. runtime_config.json (built from the ACTUAL tflite input/output details above) ─
in_d = input_details[0]
runtime_config = {
    "model_file": "fruit_classifier.tflite",
    "input": {
        "tensor_name": in_d["name"],
        "shape": [int(s) for s in in_d["shape"]],
        "layout": "NHWC",
        "dtype": str(in_d["dtype"].__name__ if hasattr(in_d["dtype"], "__name__") else in_d["dtype"]),
        "preprocessing": {
            "resize": [IMG_SIZE, IMG_SIZE],
            "resize_note": "resize (not center-crop) to IMG_SIZE x IMG_SIZE, RGB channel order",
            "rescale": "pixel values to [0, 1] by dividing by 255",
            "normalize_mean": [0.485, 0.456, 0.406],
            "normalize_std": [0.229, 0.224, 0.225],
            "normalize_note": "applied per-channel after rescale: (x - mean) / std"
        },
        "layout_warning": "PyTorch model is NCHW (1,3,384,384) but the exported .tflite input is NHWC — feed NHWC into the TFLite interpreter."
    },
    "outputs": [
        {"tensor_name": od["name"], "shape": [int(s) for s in od["shape"]]}
        for od in output_details
    ],
    "inference_notes": {
        "single_pass": "one forward pass returns all four outputs together — no separate calls needed",
        "combined_display_label": "for a single human-readable result, combine as f\'{fruit_classes[argmax(fruit_logits)]} - {condition_classes[argmax(condition_logits)]}\'"
    }
}
with open(os.path.join(EXPORT_DIR, "runtime_config.json"), "w") as f:
    json.dump(runtime_config, f, indent=2)
print("✓ runtime_config.json saved")
print(json.dumps(runtime_config, indent=2))

✓ runtime_config.json saved
{
  "model_file": "fruit_classifier.tflite",
  "input": {
    "tensor_name": "serving_default_args_0",
    "shape": [
      1,
      3,
      384,
      384
    ],
    "layout": "NHWC",
    "dtype": "float32",
    "preprocessing": {
      "resize": [
        384,
        384
      ],
      "resize_note": "resize (not center-crop) to IMG_SIZE x IMG_SIZE, RGB channel order",
      "rescale": "pixel values to [0, 1] by dividing by 255",
      "normalize_mean": [
        0.485,
        0.456,
        0.406
      ],
      "normalize_std": [
        0.229,
        0.224,
        0.225
      ],
      "normalize_note": "applied per-channel after rescale: (x - mean) / std"
    },
    "layout_warning": "PyTorch model is NCHW (1,3,384,384) but the exported .tflite input is NHWC \u2014 feed NHWC into the TFLite interpreter."
  },
  "outputs": [
    {
      "tensor_name": "serving_default_output_0_output",
      "shape": [
        1,
        5
      ]
    },
    {
      

In [10]:
import shutil, os
from google.colab import files

zip_stage = "app_dev_handover_part1"
os.makedirs(zip_stage, exist_ok=True)

shutil.copy(TFLITE_OUT, os.path.join(zip_stage, TFLITE_OUT))
shutil.copy(os.path.join(EXPORT_DIR, "class_mapping.json"), zip_stage)
shutil.copy(os.path.join(EXPORT_DIR, "runtime_config.json"), zip_stage)

shutil.make_archive("app_dev_handover_part1", "zip", zip_stage)
files.download("app_dev_handover_part1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>